In [55]:
import pandas as pd
import numpy as np

In [56]:
fights = pd.read_csv(r'C:\Users\mplan\Desktop\ufc\TRAINING_DATA\fights_training.csv')

fighters = pd.read_csv(r'C:\Users\mplan\Desktop\ufc\TRAINING_DATA\fighters_training.csv')

In [57]:
data = fights[['fighter1_id','fighter2_id','method_basic_encoded']]
data = pd.merge(data,fighters,left_on='fighter1_id',right_on='ID')
data = pd.merge(data,fighters,left_on='fighter2_id',right_on='ID')


In [58]:
data.columns

Index(['fighter1_id', 'fighter2_id', 'method_basic_encoded', 'wins_x',
       'losses_x', 'draws_x', 'SLpM_x', 'SApM_x', 'TD Avg._x', 'Sub. Avg._x',
       'ID_x', 'height_x', 'weight_kg_x', 'Str. Acc._total_x',
       'Str. Def_total_x', 'TD Acc._total_x', 'TD Def._total_x', 'reach_cm_x',
       'stance_x', 'age_x', 'wins_y', 'losses_y', 'draws_y', 'SLpM_y',
       'SApM_y', 'TD Avg._y', 'Sub. Avg._y', 'ID_y', 'height_y', 'weight_kg_y',
       'Str. Acc._total_y', 'Str. Def_total_y', 'TD Acc._total_y',
       'TD Def._total_y', 'reach_cm_y', 'stance_y', 'age_y'],
      dtype='str')

In [59]:
data['method_basic_encoded'].value_counts()
# 2 είναι το other που δεν το θέλουμε 

method_basic_encoded
3    4061
1    2825
0    1678
2     116
Name: count, dtype: int64

In [60]:
# to y
data = data[data['method_basic_encoded'] != 2]  # remove OTHER

data['method_basic_encoded'] = data['method_basic_encoded'].map({
    0: 0,   # SUB
    1: 1,   # KO/TKO
    3: 2    # DEC -> remap to 2
})

y = data['method_basic_encoded']

In [61]:
data['wins_diff'] = data['wins_x'] - data['wins_y']
data['losses_diff'] = data['losses_x'] - data['losses_y']
data['age_diff'] = data['age_x'] - data['age_y']
data['reach_diff'] = data['reach_cm_x'] - data['reach_cm_y']

data['SLpM_diff'] = data['SLpM_x'] - data['SLpM_y']
data['SApM_diff'] = data['SApM_x'] - data['SApM_y']

data['TD_diff'] = data['TD Avg._x'] - data['TD Avg._y']
data['Sub_diff'] = data['Sub. Avg._x'] - data['Sub. Avg._y']

data['StrAcc_diff'] = data['Str. Acc._total_x'] - data['Str. Acc._total_y']
data['StrDef_diff'] = data['Str. Def_total_x'] - data['Str. Def_total_y']


data['finish_rate_x'] = data['wins_x'] / (data['wins_x'] + data['losses_x'] + 1)
data['finish_rate_y'] = data['wins_y'] / (data['wins_y'] + data['losses_y'] + 1)
data['finish_diff'] = data['finish_rate_x'] - data['finish_rate_y']

data['experience_diff'] = (
    (data['wins_x'] + data['losses_x']) -
    (data['wins_y'] + data['losses_y'])
)

data['striking_gap'] = data['SLpM_diff'] * data['StrAcc_diff']
data['grappling_gap'] = data['TD_diff'] * data['Sub_diff']



In [62]:
features = [
    'wins_diff',
    'losses_diff',
    'age_diff',
    'reach_diff',

    'SLpM_diff',
    'SApM_diff',
    'TD_diff',
    'Sub_diff',

    'StrAcc_diff',
    'StrDef_diff',

    'finish_diff',
    'experience_diff',

    'striking_gap',
    'grappling_gap'
]

In [63]:
X = data[features]


In [64]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

              precision    recall  f1-score   support

           0       0.53      0.28      0.36       336
           1       0.45      0.25      0.33       565
           3       0.53      0.79      0.63       812

    accuracy                           0.51      1713
   macro avg       0.50      0.44      0.44      1713
weighted avg       0.50      0.51      0.48      1713

[[ 93  49 194]
 [ 41 144 380]
 [ 43 126 643]]


In [ ]:
from xgboost import XGBClassifier


model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

from sklearn.metrics import classification_report, confusion_matrix

# =========================
# 8. EVALUATION
# =========================
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.47      0.28      0.35       336
           1       0.48      0.34      0.40       565
           2       0.54      0.74      0.63       812

    accuracy                           0.52      1713
   macro avg       0.50      0.46      0.46      1713
weighted avg       0.51      0.52      0.50      1713



In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=500,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train, y_train)



y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.47      0.24      0.32       336
           1       0.46      0.28      0.35       565
           2       0.53      0.78      0.63       812

    accuracy                           0.51      1713
   macro avg       0.49      0.43      0.43      1713
weighted avg       0.50      0.51      0.48      1713

[[ 80  53 203]
 [ 45 158 362]
 [ 46 130 636]]
